# NST2062 Technical Assignment #2 (2026)
## Investigating Minds and Machines From the Inside

**Total: 100 points (80 + 20 bonus)**

**Estimated time: 4–8 hours**

Dear Friend, this assignment has two problems. Both use large language models as the **object of study** — not just as a tool. You will probe what models know, how they express (or fail to express) uncertainty, and how the reward signal shapes their outputs.

### Ground Rules
- You **may** use LLMs (ChatGPT, Claude, Copilot, etc.) to help you write code, debug, and brainstorm. This is expected.
- You **must** document every LLM interaction that materially shaped your solution in the **Metacognitive Reflection** sections. Prompts, what worked, what didn't, what you had to fix.
- The reflection sections are **graded** and cannot be generated by an LLM. They must be in your own voice.
- Submit: this completed notebook (.ipynb) with all cells run + outputs visible.

---

## Problem 1: Calibration & Confidence — Do Models Know What They Don't Know?
### (45 points + 10 bonus)

**Course connections**: Decisions Under Uncertainty (Bayesian brain, calibration, overconfidence), Representation (what's encoded in embedding space), Learning & Reward (RLHF shapes confidence).

**The question**: When an LLM says it's confident, should you believe it? You saw in the Calibration Game (Lecture 4) that humans are overconfident on hard questions and underconfident on easy ones. LLMs are overconfident uniformly — because RLHF rewarded confident-sounding answers. In this problem, you'll measure this empirically.

---

### Part A: Build a Calibration Dataset (10 points)

Create a dataset of **50 factual questions** with known ground-truth answers. Your dataset must include:
- 10 **easy** questions (most people would know: e.g., "What is the capital of France?")
- 15 **medium** questions (educated guess: e.g., "How many bones in the adult human body?")
- 15 **hard** questions (specialist knowledge: e.g., "What year was the Rescorla-Wagner model published?")
- 10 **unanswerable/trick** questions (no clear answer, or trick framing: e.g., "What is the weight of the colour blue?")

For each question, record:
- The question text
- The ground-truth answer (or "unanswerable" for trick questions)
- The difficulty category (easy / medium / hard / unanswerable)

**Tip**: Mix domains — some from the course content (neuroscience, AI), some general knowledge, some current events. The mix matters for the analysis.

In [ ]:
# Your code here: build the dataset as a list of dicts or a DataFrame
# Example structure:
# questions = [
#   {"question": "What is the capital of France?", "answer": "Paris", "difficulty": "easy"},
#   ...
# ]

### Part B: Measure YOUR Calibration (10 points)

Before querying any model, **answer 20 of your own questions yourself** (pick 5 from each difficulty category). For each:
1. Write your answer
2. Rate your confidence: 50% (pure guess) to 100% (certain)

Then compute:
- Your **accuracy** per difficulty category
- Your **average confidence** per difficulty category
- Your **calibration error**: |confidence - accuracy| averaged across categories

Visualise this as a **calibration plot**: x-axis = confidence bucket, y-axis = actual accuracy. A perfectly calibrated person lies on the diagonal.

**This section is personal and must be done honestly before looking at the model's answers.**

In [ ]:
# Your code here: record your own answers, compute calibration, plot

### Part C: Measure the MODEL's Calibration (15 points)

Query an LLM (use the OpenAI API, Anthropic API, or any accessible API — free tiers are fine) with all 50 questions. For each question, use the following prompt template:

```
Answer the following question. After your answer, rate your confidence
from 0% to 100% that your answer is correct. Format your response as:
Answer: [your answer]
Confidence: [X]%
```

Then:
1. **Parse** the model's answers and confidence ratings
2. **Score** each answer as correct or incorrect (you may need fuzzy matching — document your approach)
3. Compute the **model's calibration** the same way you computed yours:
   - Accuracy per difficulty category
   - Average confidence per difficulty category
   - Calibration error per category
   - Calibration plot

4. **Compare** your calibration plot to the model's calibration plot side by side.

**Key analysis questions** (answer in a markdown cell):
- Is the model overconfident, underconfident, or well-calibrated?
- Does its calibration error vary with difficulty? (Compare to humans from Lichtenstein et al. 1982)
- How does the model handle the unanswerable questions? Does it say "I don't know" or does it confabulate confidently?
- How does the model's calibration pattern differ from yours?

In [ ]:
# Your code here: query the model, parse responses, compute calibration, plot comparison

### Part D: Can You Improve the Model's Calibration? (10 points)

Try at least **two** prompt engineering interventions to improve the model's calibration. Ideas:
- Ask it to "think step by step before rating confidence"
- Ask it to "list reasons it might be wrong before answering"
- Ask it to "rate confidence on a scale of 1-10 instead of percentage"
- Provide a few-shot example of well-calibrated answers (including "I'm not sure")
- Ask in a different persona ("You are a careful scientist who hates being wrong...")

For each intervention:
1. Re-run the 50 questions with the new prompt
2. Compute calibration again
3. Compare to the baseline

**Report**: Which intervention helped most? Which didn't help? Why do you think that is (connect to what you know about RLHF and reward signals from Lecture 3)?

In [ ]:
# Your code here: implement interventions, re-run, compare

### Bonus: Token-Level Probabilities (10 bonus points)

If you're using an API that exposes log-probabilities (e.g., OpenAI with `logprobs=True`):
1. Extract the **token-level probabilities** for the model's answers
2. Compare the model's **stated** confidence (from the text) to its **actual** token probabilities
3. Are they correlated? Is the model's internal uncertainty (log-probs) better calibrated than its stated confidence?

This gets at a deep question from Lecture 2 (Representation): the model may have an internal representation of uncertainty that differs from what it expresses in text. The expressed confidence was shaped by RLHF; the log-probs were shaped by pre-training.

In [ ]:
# Your code here (bonus): extract logprobs, compare to stated confidence

### 🪞 Metacognitive Reflection — Problem 1 (graded, 5 points included above)

Write 200–400 words answering these questions. **This must be in your own voice — not generated by an LLM.**

1. Were you surprised by your own calibration results? Where were you most overconfident?
2. Were you surprised by the model's calibration? How did it compare to your expectations?
3. What did you learn about the relationship between confidence and accuracy — in yourself and in the model?
4. How did you use LLMs during this problem? What did you prompt for, what did you have to fix, and what did the LLM get wrong?
5. Connect your findings to one concept from the course (e.g., Goodhart's Law, prediction error, the Bayesian brain). How does your data illustrate or challenge that concept?

*Your reflection here (double-click to edit):*




---
## Problem 2: Temperature, Creativity, and Mode Collapse — Does Optimization Kill Novelty?
### (35 points + 10 bonus)

**Course connections**: Creativity (mode collapse, exploration vs exploitation), Learning & Reward (RLHF as reward shaping), Representation (traversal of embedding space), Decisions (temperature as the creativity dial).

**The question**: In Lecture 5 (Creativity), we argued that RLHF causes mode collapse — the model converges on safe, crowd-pleasing outputs and loses the tail distribution where novelty lives. In this problem, you'll measure this directly.

---

### Part A: The Divergent Thinking Experiment (10 points)

Use the **Alternative Uses Task** (Guilford, 1967) — the same test you did in the Creativity lecture.

Pick **3 common objects** (e.g., brick, paperclip, shoe). For each object, prompt the LLM to:
> "List 20 unusual uses for a [object]. Be creative and surprising."

Run this at **5 different temperature settings**: 0.0, 0.3, 0.7, 1.0, 1.5 (or as close as the API allows).

For each (object × temperature) combination, record all 20 responses.

That gives you 3 objects × 5 temperatures × 20 uses = 300 data points.

In [ ]:
# Your code here: run the divergent thinking experiment across temperatures

### Part B: Measuring Creativity — Design Your Own Rubric (10 points)

This is the hard part. There is no standard metric for creativity (that was the Evaluation Problem from Lecture 5). You need to **design a scoring rubric** with at least 3 dimensions. Suggested dimensions (pick 3-4, or invent your own):

- **Originality**: How surprising/uncommon is the use? (1 = obvious, 5 = never heard of it)
- **Feasibility**: Could you actually do this? (1 = impossible, 5 = easy)
- **Humour**: Is it funny or playful? (1 = not at all, 5 = genuinely funny)
- **Elaboration**: How detailed/specific is the response? (1 = generic, 5 = vivid)
- **Category diversity**: Does the list span different domains (physical, social, artistic, absurd)?

**Important**: Score a random sample of **5 uses per temperature** per object (75 total). Do this yourself — do NOT use an LLM to evaluate creativity.

Then compute and plot:
1. **Average score per dimension** as a function of temperature
2. **Lexical diversity** (number of unique words / total words) as a function of temperature
3. **Semantic diversity** (average pairwise cosine distance between sentence embeddings) as a function of temperature

For semantic diversity, use a sentence embedding model (e.g., `sentence-transformers/all-MiniLM-L6-v2` from HuggingFace).

In [ ]:
# Your code here: implement rubric, score samples, compute diversity metrics, plot

### Part C: Detecting Mode Collapse (15 points)

Now test whether RLHF narrows the output distribution. Design an experiment:

1. **Repetition test**: Run the same prompt 10 times at temperature 0.7. How many of the 20 uses are **identical** across runs? How many are **semantically similar** (cosine similarity > 0.85)? A model in mode collapse will repeat itself; a diverse model won't.

2. **Safe vs. surprising**: Classify each use as **safe** (conventional, predictable — e.g., "use a brick as a doorstop") or **surprising** (unexpected, creative — e.g., "use a brick as a pillow to build character"). Plot the ratio of safe:surprising as a function of temperature.

3. **Compare models (if possible)**: If you have access to both a base model and an RLHF'd model (e.g., via different API endpoints or open-source models like Llama base vs. Llama-chat), compare their diversity at the same temperature. The prediction from Lecture 5: the RLHF'd model should be less diverse.

**Analysis questions** (answer in a markdown cell):
- At what temperature does the model produce the best balance of quality and originality?
- Is there evidence of mode collapse? Where?
- How does this connect to the exploration/exploitation tradeoff from the Decisions lecture?
- If you compared models: does RLHF reduce diversity as predicted?

In [ ]:
# Your code here: repetition test, safe vs surprising classification, optional model comparison

### Bonus: The Audience Matters (10 bonus points)

**Course connection**: Mentalising (Lecture 6) — creativity needs an audience.

Add audience context to the prompt and measure whether it changes the outputs:
> "List 20 unusual uses for a brick. Your audience is [a group of 5-year-olds / a panel of art critics / a team of engineers / a comedy club audience]."

For each audience:
1. Run at temperature 0.7
2. Score using your rubric
3. Measure how the output distribution shifts

**Analysis**: Does the model adapt its creativity to the audience? Is it doing something like mentalising — modelling what each audience would find creative? Or is it just pattern-matching on audience-associated vocabulary?

In [ ]:
# Your code here (bonus): audience experiment

### 🪞 Metacognitive Reflection — Problem 2 (graded, 5 points included above)

Write 200–400 words answering these questions. **This must be in your own voice — not generated by an LLM.**

1. What was the hardest part of designing a creativity rubric? What did it teach you about the evaluation problem?
2. Did the temperature results match your predictions? What surprised you?
3. Did you find evidence of mode collapse? If yes, is this a problem — or is it the model being appropriately safe?
4. How did you use LLMs during this problem? Be specific about what you prompted for and what you had to do yourself.
5. Connect your findings to the course: if creativity is search through representation space, and temperature controls the search width, what does your data tell you about the shape of that space?

*Your reflection here (double-click to edit):*




---
## Grading Summary

| Component | Points |
|---|---|
| **Problem 1** | |
| Part A: Calibration dataset | 10 |
| Part B: Your own calibration | 10 |
| Part C: Model calibration + comparison | 15 |
| Part D: Prompt interventions | 10 |
| Bonus: Token-level probabilities | (+10) |
| **Problem 2** | |
| Part A: Divergent thinking experiment | 10 |
| Part B: Creativity rubric + metrics | 10 |
| Part C: Mode collapse detection | 15 |
| Bonus: Audience experiment | (+10) |
| **Metacognitive Reflections** | |
| Reflection 1 (included in P1) | (5 of P1) |
| Reflection 2 (included in P2) | (5 of P2) |
| **Total** | **80 + 20 bonus** |

---

### Assessment Criteria

**Code quality** (30%): Does the code run? Is it clean and documented? Are the experiments reproducible?

**Analysis quality** (40%): Are the visualisations clear? Are the comparisons meaningful? Do the analysis questions receive thoughtful, evidence-based answers that connect to course concepts?

**Metacognitive reflections** (20%): Are the reflections genuine, specific, and self-aware? Do they demonstrate learning — not just completion?

**Rigour** (10%): Sample sizes, statistical thinking, awareness of limitations. You don't need p-values, but you should know when your sample is too small to draw strong conclusions.

---

### A Note on Using LLMs

You will use LLMs both as a **tool** (to help you code) and as the **object of study** (the thing you're investigating). This is intentional. The metacognitive reflections ask you to distinguish between these two roles. The ability to use a tool critically while simultaneously studying it is a core skill for working with AI systems — and it's a form of mentalising: modelling the capabilities and limitations of a non-human agent you're interacting with.

Have fun! I am looking forward to see your results!